# 3.0 Retrieval – BM25 + Dense Baseline

**Tác giả:** Đăng (TV1)  
**Mục tiêu:**
- Build chunks từ syllabi JSON
- Build BM25 index + Dense index (multilingual-e5-small)
- Demo search 5 câu hỏi mẫu
- Evaluate trên synthetic 80 QA: Recall@1,3,5,10, MRR@10
- Lưu chunks.jsonl + vectors.npy

In [ ]:
import json
import sys
import random
import math
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path('..').resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.syllabus_core.retrieval.index import BM25Index, DenseIndex, build_chunks

import yaml
with open(ROOT / 'config' / 'params.yaml') as f:
    CFG = yaml.safe_load(f)

RANDOM_SEED = CFG.get('random_seed', 42)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

FIG_DIR = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = ROOT / 'data' / 'processed'
METRICS_PATH = ROOT / 'results' / 'metrics.json'

BM25_TOP_K = CFG.get('m3_retrieval', {}).get('bm25_top_k', 30)
DENSE_TOP_K = CFG.get('m3_retrieval', {}).get('dense_top_k', 30)
CONTEXT_K = CFG.get('m3_retrieval', {}).get('context_k', 5)
EMBED_MODEL = CFG.get('m3_embedding', {}).get('model', 'intfloat/multilingual-e5-small')

print(f'BM25_TOP_K={BM25_TOP_K}, DENSE_TOP_K={DENSE_TOP_K}, CONTEXT_K={CONTEXT_K}')
print(f'Embedding model: {EMBED_MODEL}')

## 1. Build chunks từ syllabi

In [ ]:
SYLLABI_DIR = PROCESSED_DIR / 'syllabi'

syllabi = []
if SYLLABI_DIR.exists():
    for fp in sorted(SYLLABI_DIR.glob('*.json')):
        try:
            with open(fp, encoding='utf-8') as f:
                syllabi.append(json.load(f))
        except Exception:
            pass

if not syllabi:
    print('Không có syllabi thật → dùng dữ liệu tổng hợp 10 môn')
    SECTION_NAMES = [
        'general_info', 'course_description', 'course_objectives',
        'course_learning_outcomes', 'course_content', 'learning_resources',
        'teaching_schedule', 'course_policies', 'assessment'
    ]
    COURSE_DATA = [
        ('841020', 'Cơ sở lập trình', 3),
        ('841303', 'Cấu trúc dữ liệu', 4),
        ('841109', 'Mạng máy tính', 3),
        ('841434', 'Cơ sở dữ liệu', 3),
        ('841470', 'Trí tuệ nhân tạo', 3),
        ('841048', 'Giải tích 1', 3),
        ('841323', 'Đại số tuyến tính', 3),
        ('841462', 'Lập trình hướng đối tượng', 3),
        ('841479', 'Kỹ nghệ phần mềm', 3),
        ('841479', 'Hệ điều hành', 3),
    ]
    for code, name, credits in COURSE_DATA:
        syllabi.append({
            'course_code': code,
            'course_name': name,
            'credits': credits,
            'prerequisites': [],
            'clos': [{'clo_id': 'CLO1', 'text': f'Áp dụng kiến thức {name} giải bài toán thực tế', 'bloom_level': 3}],
            'sections': {
                s: f'Nội dung mục {s} của môn {name} (mã {code}). '
                   f'Môn học {name} có {credits} tín chỉ cung cấp nền tảng về {name.lower()}.'
                for s in SECTION_NAMES
            },
            'missing_sections': [],
        })

# Build chunks
chunks = build_chunks(syllabi)
print(f'Đã build {len(chunks)} chunks từ {len(syllabi)} syllabi')
print(f'Ví dụ chunk 0:')
print(json.dumps(chunks[0], ensure_ascii=False, indent=2))

## 2. Build BM25 Index + Dense Index

In [ ]:
# BM25 Index
bm25_idx = BM25Index()
bm25_idx.build(chunks)
print(f'BM25 index: {bm25_idx.n_docs} docs')

In [ ]:
# Dense Index
dense_idx = DenseIndex(model_name=EMBED_MODEL)
try:
    dense_idx.build(chunks)
    print(f'Dense index: {dense_idx.n_docs} docs, dim={dense_idx.dim}')
except Exception as e:
    print(f'Dense index lỗi: {e}')
    print('Sẽ dùng TF-IDF cosine thay thế trong evaluate')

In [ ]:
# Lưu chunks.jsonl
chunks_path = PROCESSED_DIR / 'chunks.jsonl'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
with open(chunks_path, 'w', encoding='utf-8') as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')
print(f'Đã lưu {len(chunks)} chunks → {chunks_path}')

# Lưu vectors nếu có
if hasattr(dense_idx, 'vectors') and dense_idx.vectors is not None:
    np.save(PROCESSED_DIR / 'vectors.npy', dense_idx.vectors)
    print(f'Đã lưu vectors.npy: shape={dense_idx.vectors.shape}')

## 3. Demo search 5 câu hỏi

In [ ]:
DEMO_QUERIES = [
    'Môn Cơ sở lập trình có bao nhiêu tín chỉ?',
    'Chuẩn đầu ra của môn Trí tuệ nhân tạo là gì?',
    'Môn nào yêu cầu tiên quyết Giải tích 1?',
    'Phương pháp đánh giá của môn Cấu trúc dữ liệu?',
    'Tài liệu học tập của môn Mạng máy tính?',
]

print('=== DEMO BM25 SEARCH ===')
for q in DEMO_QUERIES:
    results = bm25_idx.search(q, top_k=3)
    print(f'Q: {q}')
    for r in results:
        print(f'  [{r["score"]:.3f}] {r["text"][:80]}')
    print()

## 4. Evaluate trên synthetic 80 QA pairs

In [ ]:
# Sinh 80 QA pairs tổng hợp
rng = random.Random(RANDOM_SEED)
qa_pairs = []
templates = [
    ('Môn {name} có bao nhiêu tín chỉ?', 'general_info'),
    ('Chuẩn đầu ra của môn {name}?', 'course_learning_outcomes'),
    ('Mô tả môn học {name}?', 'course_description'),
    ('Mục tiêu môn {name}?', 'course_objectives'),
    ('Nội dung môn {name}?', 'course_content'),
    ('Tài liệu học môn {name}?', 'learning_resources'),
    ('Phương pháp đánh giá môn {name}?', 'assessment'),
    ('Quy định lớp học môn {name}?', 'course_policies'),
]

for syl in syllabi:
    code = syl['course_code']
    name = syl['course_name']
    for tmpl, section in rng.choices(templates, k=max(1, 80 // len(syllabi))):
        q = tmpl.format(name=name)
        # Relevant chunk: chunk có course_code==code và section==section
        relevant_ids = [
            c['chunk_id'] for c in chunks
            if c.get('course_code') == code and c.get('section') == section
        ]
        if relevant_ids:
            qa_pairs.append({'question': q, 'relevant_chunk_ids': relevant_ids, 'course_code': code})

# Đảm bảo đủ 80
while len(qa_pairs) < 80:
    syl = rng.choice(syllabi)
    tmpl, section = rng.choice(templates)
    q = tmpl.format(name=syl['course_name'])
    relevant_ids = [c['chunk_id'] for c in chunks
                    if c.get('course_code') == syl['course_code'] and c.get('section') == section]
    if relevant_ids:
        qa_pairs.append({'question': q, 'relevant_chunk_ids': relevant_ids})

qa_pairs = qa_pairs[:80]
print(f'Sinh {len(qa_pairs)} QA pairs')

In [ ]:
def evaluate_retrieval(index, qa_pairs, top_ks=(1, 3, 5, 10)):
    """Tính Recall@k và MRR@10 cho retrieval index."""
    metrics = {f'recall@{k}': 0.0 for k in top_ks}
    mrr = 0.0
    for qa in qa_pairs:
        q = qa['question']
        rel = set(qa['relevant_chunk_ids'])
        results = index.search(q, top_k=max(top_ks))
        retrieved_ids = [r['chunk_id'] for r in results]
        # Recall@k
        for k in top_ks:
            if any(rid in rel for rid in retrieved_ids[:k]):
                metrics[f'recall@{k}'] += 1
        # MRR
        for rank, rid in enumerate(retrieved_ids[:10], start=1):
            if rid in rel:
                mrr += 1.0 / rank
                break
    n = len(qa_pairs)
    for k in metrics:
        metrics[k] = round(metrics[k] / n, 4)
    metrics['mrr@10'] = round(mrr / n, 4)
    return metrics


# Evaluate BM25
bm25_metrics = evaluate_retrieval(bm25_idx, qa_pairs)
print('BM25:', bm25_metrics)

# Evaluate Dense (nếu có)
try:
    dense_metrics = evaluate_retrieval(dense_idx, qa_pairs)
    print('Dense:', dense_metrics)
except Exception as e:
    print(f'Dense eval lỗi: {e}')
    dense_metrics = {k: 0.0 for k in bm25_metrics}

In [ ]:
with open(METRICS_PATH, encoding='utf-8') as f:
    metrics = json.load(f)

for k, v in bm25_metrics.items():
    metrics['m3_retrieval']['configs']['bm25_only'][k.replace('@', '_at_').replace('mrr_at', 'mrr_at')] = v
metrics['m3_retrieval']['configs']['bm25_only']['status'] = 'completed'

for k, v in dense_metrics.items():
    metrics['m3_retrieval']['configs']['dense_only'][k.replace('@', '_at_').replace('mrr_at', 'mrr_at')] = v
metrics['m3_retrieval']['configs']['dense_only']['status'] = 'completed'

# Lưu QA pairs
qa_path = PROCESSED_DIR / 'synthetic_qa_80.json'
with open(qa_path, 'w', encoding='utf-8') as f:
    json.dump(qa_pairs, f, ensure_ascii=False, indent=2)

with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print('BM25 metrics:', bm25_metrics)
print('Dense metrics:', dense_metrics)
print('Đã lưu metrics và QA pairs')